In [ ]:
# Ensure PyTorch is available in this notebook kernel
import importlib.util
import subprocess
import sys

from pathlib import Path

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])

import torch
print("torch:", torch.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 4.0 MB/s eta 0:00:001
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 550.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 862.0 kB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/minhcao/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/minhcao/Library/Python/3.11/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "

torch: 2.2.2


In [2]:
from myAPI.model import MNISTDataset, MyData, CNN, device

import torch

import json

# device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [3]:
# Load model config
from pathlib import Path

configFile = Path("sampleModel") / "config.json"
with open(configFile, "r") as f:
    config = json.load(f)


In [10]:
# create a model
model = CNN(**config["model"]).to(device)

# create mnist dataset path is the folder that contain all four files train = false to get the test data
data_path = Path("mydata") / "Minh Cao"
mndata = MNISTDataset(data_path, train=True)
# data = MyData()

In [11]:
# use the training_process method to train, pass in the dataset and the training config
# training process returns the loss history and acc_history
[loss_hist, acc_hist] = model.training_process(mndata,**config["training"])

In [12]:
print(loss_hist)

[2.154894880263641, 1.1598492867749102, 0.5454469090512364, 0.40353200816774315, 0.3276454568813395]


In [13]:
# testing_process takes in the data and returns loss, accuracy, conf_matrix,
# wrong indices, and (optionally) wrong_details with true/pred labels

testMndata = MNISTDataset(path=data_path, train=False)
[loss, acc, conf_mat, wrong_idx, wrong_details] = model.testing_process(
    testMndata,
    return_wrong_details=True,
)


In [14]:
print(conf_mat)

tensor([[28,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0, 28,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  1, 28,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  2, 20,  0,  4,  0,  0,  1,  2],
        [ 0,  2,  1,  0, 24,  0,  2,  0,  0,  0],
        [ 0,  0,  0,  0,  3, 24,  1,  0,  0,  0],
        [ 0,  1,  0,  0,  1,  0, 26,  0,  1,  0],
        [ 0,  0,  0,  0,  1,  0,  0, 25,  3,  0],
        [ 0,  0,  0,  0,  0,  2,  0,  2, 24,  0],
        [ 0,  0,  0,  1,  0,  0,  0,  0,  0, 28]])


In [15]:
# --- Export results (train/test loss, confusion matrix, wrong samples) ---
from myAPI.results_export import export_results

export_dir = export_results(
    out_dir="exports",
    train_loss_history=loss_hist,
    train_accuracy_history=acc_hist,
    test_loss=loss,
    test_accuracy=acc,
    confusion_matrix=conf_mat,
    wrong_indices=wrong_idx,
    wrong_details=wrong_details,
    train_dataset=mndata,
    test_dataset=testMndata,
    extra={"config": config, "device": device},
)
print("Exported to:", export_dir)


Exported to: exports/run_20260509_130900


In [ ]:
from pathlib import Path

weightFile = Path("sampleModel") / "model_weights.pth"
model.load_weight(weightFile)
model.clear_weight()
model.save_weight(Path("newWeigh.pth"))
